# 02 · Ventanas y splits

Apila las ventanas deslizantes `X`/`Y`, parte el dataset por fecha con embargo, ajusta el escalado con train y deja `data/processed/ventanas.npz`, que es el único fichero que leen los notebooks 03 a 14.

**Responsable:** Oscar

**Entradas**

- `data/processed/canales.parquet`
- `data/processed/objetivos.parquet`
- `data/processed/regimenes.parquet`

**Salidas**

- `data/processed/ventanas.npz`

**Tiempo estimado:** <1 min en CPU.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd

from src import regimenes, ventanas

canales = pd.read_parquet(src.DIR_PROCESADO / "canales.parquet")
objetivos = pd.read_parquet(src.DIR_PROCESADO / "objetivos.parquet")
tabla_regimenes = pd.read_parquet(src.DIR_PROCESADO / "regimenes.parquet")

print("canales:", canales.shape, "· objetivos:", objetivos.shape,
      "· regímenes:", tabla_regimenes.shape)

## Geometría de las ventanas

`X` cubre los 60 días de mercado anteriores a `t` inclusive; `Y` describe los 21
días posteriores. La ventana que termina en `t` nunca contiene información de
`t+1` en adelante: la separación entre entrada y objetivo es exacta, sin solape de
un solo día.

In [ ]:
v = config.ventanas()
n_regimenes = config.n_regimenes()

conjunto = ventanas.construir_ventanas(
    canales=canales,
    regimen_futuro=tabla_regimenes["regimen_futuro"],
    vol_futura=objetivos["vol_futura"],
    ventanas=v,
)

print(conjunto)
print("X:", conjunto.X.shape, "· solape máximo entre ventanas:", v.solape_maximo, "días")

## Split temporal con embargo

Las ventanas consecutivas comparten 59 de sus 60 días. Un `train_test_split`
aleatorio colocaría ventanas casi idénticas a ambos lados del corte y el modelo
obtendría métricas excelentes por memorizar. Los notebooks guiados del máster usan
split aleatorio; aquí no, y es una decisión deliberada.

Cortar por fecha tampoco basta: la última ventana de train tiene su `Y` calculada
sobre días que ya pertenecen a validación. El embargo de 85 días naturales
—por encima del mínimo teórico de `pasado + horizonte = 81`— descarta las ventanas
que cruzan un corte.

In [ ]:
p = config.particiones()

train, val, test = ventanas.partir(
    conjunto,
    train_hasta=p.train_hasta,
    val_hasta=p.val_hasta,
    embargo_dias=p.embargo_dias,
)

for nombre, parte in [("train", train), ("val", val), ("test", test)]:
    print(f"{nombre:6s} {len(parte):5d} ventanas · "
          f"{parte.fechas[0].date()} → {parte.fechas[-1].date()}")

## Verificación del embargo

El hueco real entre particiones debe superar el mínimo exigido. Si no lo hiciera,
alguna observación aparecería en dos particiones y todas las métricas de test
estarían infladas.

In [ ]:
hueco_tv = (val.fechas[0] - train.fechas[-1]).days
hueco_vt = (test.fechas[0] - val.fechas[-1]).days

print("Hueco train → val:", hueco_tv, "días naturales")
print("Hueco val → test: ", hueco_vt, "días naturales")
print("Mínimo exigido:   ", p.embargo_dias)

assert hueco_tv >= p.embargo_dias and hueco_vt >= p.embargo_dias, "El embargo no se respeta."
print("Embargo correcto.")

## Escalado ajustado solo con train

Media y desviación se calculan **por canal** y usando exclusivamente el tramo de
entrenamiento. Por canal y no por posición temporal: un retorno diario y un
z-score del VIX no son magnitudes comparables, pero dentro de un canal todas las
posiciones de la ventana comparten escala.

Los mismos estadísticos los usan el modelo downstream y los siete generadores, de
modo que todo el proyecto vive en el mismo espacio normalizado.

In [ ]:
media, desviacion = ventanas.ajustar_escalado(train)

train_e = ventanas.escalar(train, media, desviacion)
val_e = ventanas.escalar(val, media, desviacion)
test_e = ventanas.escalar(test, media, desviacion)

print("media  train escalado:", round(float(train_e.X.mean()), 6))
print("desv.  train escalado:", round(float(train_e.X.std()), 6))
print("media  test  escalado:", round(float(test_e.X.mean()), 4),
      "(no tiene por qué ser 0: el escalador no vio este tramo)")

## Reparto de clases por partición

Esta tabla justifica el taller entero: cuantifica cuán minoritaria es la clase de
crisis y, por tanto, cuánto margen hay para que los datos sintéticos aporten algo.
También avisa de un problema real: si una partición no contuviera ninguna ventana
de crisis, las métricas de esa clase serían indefinidas.

In [ ]:
reparto = pd.concat(
    {
        nombre: regimenes.distribucion(parte.y_reg, n_regimenes)["porcentaje"]
        for nombre, parte in [("train", train), ("val", val), ("test", test)]
    },
    axis=1,
)
display(reparto)

for nombre, parte in [("train", train), ("val", val), ("test", test)]:
    print(nombre, "· NaN en X:", int(np.isnan(parte.X).sum()),
          "· NaN en y_vol:", int(np.isnan(parte.y_vol).sum()))

## El bloque que consumen los generadores

Los generadores no modelan `X` e `Y` por separado: modelan el bloque conjunto
`[X aplanada ‖ y_vol]`, de forma que cada muestra sintética viene con su objetivo
de volatilidad coherente. El régimen queda fuera del bloque porque es la
condición, no algo que el generador deba reproducir; así la etiqueta de una
muestra sintética es exacta y se puede pedir "500 ventanas de crisis".

In [ ]:
bloque = ventanas.empaquetar(train_e)

print("bloque:", bloque.shape)
print("d = pasado × canales + 1 =", v.pasado, "×", config.n_canales(), "+ 1 =",
      ventanas.dimension_bloque(v))

reconstruido = ventanas.desempaquetar(bloque, train_e.y_reg, v)
print("ida y vuelta sin pérdida:", np.allclose(reconstruido.X, train_e.X))

## Escritura

Un único `.npz` comprimido con las tres particiones ya escaladas y el escalador.
Lo escribe `ventanas.guardar_procesado()` y lo lee `ventanas.cargar_procesado()`:
el formato del fichero vive en un solo sitio, así que no puede desincronizarse
entre quien escribe y quien lee.

Es el contrato con los notebooks 03 a 14: ninguno vuelve a tocar los parquet ni
recalcula ventanas.

In [ ]:
particion = ventanas.Particion(
    train=train_e,
    val=val_e,
    test=test_e,
    media=media,
    desviacion=desviacion,
)
ruta = ventanas.guardar_procesado(particion)

print("Escrito", ruta.name, "·", round(ruta.stat().st_size / 1e6, 1), "MB")

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_PROCESADO / "ventanas.npz",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
